# KungfuBot — G1 功夫动作 MuJoCo 一键预览

基于 [TeleHuman/PBHC](https://github.com/TeleHuman/PBHC) (KungfuBot) 的预训练策略，在 **MuJoCo** 中驱动 **Unitree G1** 完成中国功夫动作（马步、马步出拳等）。

## 训练 → 部署 流程

```
人体视频/AMASS → SMPL → 重定向到 G1 → IsaacGym 训练 RL 策略 → 导出 ONNX → MuJoCo sim2sim
                                                                              ↑ 本 Notebook 从这里开始
```

我们直接使用仓库自带的 3 个预训练 ONNX，**跳过训练**，在 MuJoCo 中部署。

| 动作 | ONNX 权重 | 参考动作 |
|---|---|---|
| 🥋 马步姿态 (v1) | `pretrained_horse_stance_pose/exported/model_50000.onnx` | `Horse-stance_pose.pkl` |
| 🥋 马步姿态 (v2) | `pretrained_horse_stance_pose_2/exported/model_119000.onnx` | `Horse-stance_pose.pkl` |
| 👊 **马步出拳** | `pretrained_horse_stance_punch/exported/model_33000.onnx` | `Horse-stance_punch.pkl` |

---

## 1. 初始化子模块

In [ ]:
!git submodule update --init --recursive dependencies/KungfuBot

## 2. 创建并安装 conda 环境（幂等，一键完成）

官方要求 Python 3.8（PBHC 把 `numpy==1.24.4` / `mujoco==3.2.3` 等 pin 死了）。

> ⚠️ **不要复用本项目的 `mujoco` 环境**：PBHC 的 pin 会**降级** `mujoco` 环境里的 numpy 2.x 与 mujoco 3.8.1，连带把 `DEMO.ipynb` / `UNITREE.ipynb` 等其它演示弄坏。这里用独立的 `kungfubot` 环境。

> 如果只跑 MuJoCo 预览，**不需要安装 IsaacGym**（IsaacGym 仅训练阶段需要）。

下面这一格幂等：env 已存在则跳过创建，pip 也只补齐缺失部分。**先跑这一格，再跑后面的预览。**

In [ ]:
%%bash
set -e
source "$(conda info --base)/etc/profile.d/conda.sh"

ENV=kungfubot
if conda env list | awk '{print $1}' | grep -qx "$ENV"; then
    echo "✅ conda env '$ENV' already exists, skipping create."
else
    echo "📦 creating conda env '$ENV' (python=3.8) ..."
    conda create -n "$ENV" python=3.8 -y
fi

echo "📦 installing KungfuBot (PBHC) ..."
conda run -n "$ENV" --no-capture-output pip install -e ./dependencies/KungfuBot
echo "📦 installing isaac_utils (纯运动学库) ..."
conda run -n "$ENV" --no-capture-output pip install -e ./dependencies/KungfuBot/humanoidverse/isaac_utils

# PBHC setup.py 漏写了 torch 依赖,但 urci.py / urcirobot.py 都 import torch,这里补装
echo "📦 installing torch (PBHC setup.py missing) ..."
conda run -n "$ENV" --no-capture-output pip install torch

conda run -n "$ENV" --no-capture-output pip install -q ipykernel
conda run -n "$ENV" python -m ipykernel install --user --name="$ENV" --display-name "Python ($ENV)" >/dev/null 2>&1 || true

echo "🎉 done. 后面的格子用 conda run -n $ENV 直接跑。"

**setup.py 已声明的关键依赖：** `mujoco==3.2.3`、`mujoco-python-viewer`、`onnx`、`onnxruntime`、`hydra-core`、`numpy==1.24.4`、`scipy`、`loguru`、`dm_control`、`mink` …

---

## 3. 🚀 一键启动：G1 马步出拳 (Horse-stance Punch)

`urci.py` 会自动从 ONNX 同级目录加载 `config.yaml`（其中已写好 motion_file 与 g1 模型 xml），所以命令极简：

- `+opt=record`：开启录制 / 评估模式
- `+simulator=mujoco`：选用 MuJoCo sim2sim
- `+checkpoint=...onnx`：策略权重

`conda run` 关键参数:
- `--cwd dependencies/KungfuBot`：免去 `cd`
- `--no-capture-output`：实时输出日志（默认会缓冲到进程结束，MuJoCo 日志会看不到）

运行后会弹出 MuJoCo 可视化窗口，G1 摆好马步并出拳。

In [ ]:
!conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python humanoidverse/urci.py \
        +opt=record \
        +simulator=mujoco \
        +checkpoint=example/pretrained_horse_stance_punch/exported/model_33000.onnx

### MuJoCo 窗口按键

见 `dependencies/KungfuBot/humanoidverse/deploy/mujoco.py` 中的 `MViewerPlugin`。常用：

| 按键 | 功能 |
|---|---|
| `Space` | 暂停 / 继续 |
| `Backspace` | 重置场景 |
| `R` | 重放参考动作 |
| `0` `1` `2` … | 多策略模式下切换策略 |
| 鼠标拖拽 | 对机器人施加外力 / 旋转视角 |

---

## 4. 其他动作预览

### 🥋 马步姿态（v1，5w 步）

In [ ]:
!conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python humanoidverse/urci.py \
        +opt=record \
        +simulator=mujoco \
        +checkpoint=example/pretrained_horse_stance_pose/exported/model_50000.onnx

### 🥋 马步姿态（v2，11.9w 步，更稳定）

In [ ]:
!conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python humanoidverse/urci.py \
        +opt=record \
        +simulator=mujoco \
        +checkpoint=example/pretrained_horse_stance_pose_2/exported/model_119000.onnx

---

## 5. 只看参考动作（无需策略 / 无需训练）

如果你只想可视化 `.pkl` 中的人体重定向轨迹（看「目标动作长啥样」），用仓库自带的 MuJoCo 查看器：

支持的 `.pkl`（位于 `dependencies/KungfuBot/example/motion_data/`）：

- `Bruce_Lee_pose.pkl` ⭐ 李小龙姿态
- `Charleston_dance.pkl` 查尔斯顿舞
- `Hooks_punch.pkl` 勾拳
- `Horse-stance_pose.pkl` 马步
- `Horse-stance_punch.pkl` 马步出拳
- `Roundhouse_kick.pkl` 回旋踢
- `Side_kick.pkl` 侧踢

⚠️ 这些只有 `Horse-stance_pose` / `Horse-stance_punch` 自带预训练策略，其余动作要看「策略跟随效果」需自行在 IsaacGym 训练后导出 ONNX。

### 7 个参考动作 — 每个一键预览

_7 reference motions — one-click preview each_

按 `Space` 暂停 / `R` 重置 / `Backspace` 重启。

In [ ]:
# 李小龙姿态 / Bruce Lee pose
!conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python robot_motion_process/vis_q_mj.py \
        +motion_file=example/motion_data/Bruce_Lee_pose.pkl

In [ ]:
# 查尔斯顿舞 / Charleston dance
!conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python robot_motion_process/vis_q_mj.py \
        +motion_file=example/motion_data/Charleston_dance.pkl

In [ ]:
# 勾拳 / Hooks punch
!conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python robot_motion_process/vis_q_mj.py \
        +motion_file=example/motion_data/Hooks_punch.pkl

In [ ]:
# 马步 / Horse-stance pose
!conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python robot_motion_process/vis_q_mj.py \
        +motion_file=example/motion_data/Horse-stance_pose.pkl

In [ ]:
# 马步出拳 / Horse-stance punch
!conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python robot_motion_process/vis_q_mj.py \
        +motion_file=example/motion_data/Horse-stance_punch.pkl

In [ ]:
# 回旋踢 / Roundhouse kick
!conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python robot_motion_process/vis_q_mj.py \
        +motion_file=example/motion_data/Roundhouse_kick.pkl

In [ ]:
# 侧踢 / Side kick
!conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python robot_motion_process/vis_q_mj.py \
        +motion_file=example/motion_data/Side_kick.pkl

---

## 6. 自定义：在你自己的 pkl + onnx 上部署

训完一个新动作（例如 `Roundhouse_kick`）后，部署命令：

```bash
conda run -n kungfubot --cwd dependencies/KungfuBot --no-capture-output \
    python humanoidverse/urci.py \
        +opt=record \
        +simulator=mujoco \
        +checkpoint=<path/to/your.onnx> \
        +robot.motion.motion_file=example/motion_data/Roundhouse_kick.pkl \
        +robot.asset.xml_file=g1/g1_23dof_lock_wrist_rev_2.xml
```

训练命令见 `dependencies/KungfuBot/humanoidverse/README.md`（需要 IsaacGym + CUDA）。

---

## 7. 常见问题

**Q: `EnvironmentNameNotFound: Could not find conda environment: kungfubot`**  
你跳过了 §2 的安装格。回去跑那一格，它幂等且会自动建 env。

**Q: 能不能直接用 `conda run -n mujoco` 复用现成的 `mujoco` 环境？**  
**不行**。PBHC `setup.py` pin 死了 `numpy==1.24.4` / `mujoco==3.2.3`，装进 `mujoco` env 会把 numpy 2.x → 1.24.4、mujoco 3.8.1 → 3.2.3 全部降级，连带打坏 `DEMO.ipynb` / `UNITREE.ipynb` 等其它演示。独立 env 是刚需。

**Q: `conda run` 没有实时输出？**  
记得加 `--no-capture-output`，否则 stdout 会缓冲到进程结束才一次性吐出来。

**Q: 窗口黑屏 / MuJoCo viewer 不弹**  
需要 GLFW + 桌面环境。在 SSH 远端跑要 `export DISPLAY=:0` 并配 X-forwarding，或改用 headless 录制（修改 `headless: true`）。

**Q: 报 `isaacgym` 缺失**  
本 Notebook 路径不应触发 IsaacGym。若报错，确认命令是 `urci.py`（部署）而非 `train_agent.py`/`eval_agent.py`（训练/IsaacGym）。

**Q: 想看 Bruce Lee 姿态的 RL 策略？**  
仓库**没有**为它提供预训练 ONNX。要么自己训，要么先用 `vis_q_mj.py` 看参考轨迹（见 §5）。

---

## 参考

- 上游仓库: [TeleHuman/PBHC](https://github.com/TeleHuman/PBHC)
- 论文方法: ASAP（Aligning Simulation and Real-World Physics for Learning Agile Humanoid Whole-Body Skills）
- 本项目其它人形 RL 部署: 见 `UNITREE.ipynb`（unitree_rl_gym 官方策略）